Ejercicio 1

In [1]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering

# Datos de entrada: nivel y horas jugadas por usuario
data = pd.DataFrame({'nivel': [12, 14, 50, 52, 90, 88], 'horas': [5, 8, 40, 45, 120, 115]})

# 1. Instancia el modelo para obtener exactamente 3 clusters
modelo = AgglomerativeClustering(n_clusters=3)

# 2. Obtén las etiquetas asignadas a cada muestra de data
etiquetas = modelo.fit_predict(data)

print("Etiquetas asignadas:", etiquetas)

Etiquetas asignadas: [2 2 1 1 0 0]


Ejercicio 2

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

X = np.array([[1, 2], [1, 4], [1, 0], [10, 2], [10, 4], [10, 0]])
silhouette_scores = []

for k in range(2, 5):
    # Instancia K-Means con k clusters y random_state=42
    kmeans = KMeans(n_clusters=k, random_state=42)
    labels = kmeans.fit_predict(X)

    # Calcula el coeficiente de silueta pasándole los datos y las etiquetas obtenidas
    score = silhouette_score(X, labels)
    silhouette_scores.append(score)

print("Silhouette Scores:", silhouette_scores)

Silhouette Scores: [0.7133477791749615, 0.43630092057240805, 0.16666666666666666]


Ejercicio 4

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

X = np.array([[100, 0.1], [200, 0.2], [150, 0.15], [300, 0.3], [500, 0.5]])

# 1. Estandariza los datos con media 0 y varianza unitaria
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Instancia PCA para reducir a 2 componentes principales
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print("Ratio de varianza explicada por cada componente:", pca.explained_variance_ratio_)

Ratio de varianza explicada por cada componente: [1.00000000e+00 1.10933565e-32]


Ejercicio 5

Resultados: [0.65, 0.20, 0.08, 0.03, 0.01, 0.01, 0.01, 0.005, 0.003, 0.002]

1 componente = 65%
2 componemntes = 85%
3 componentes = 93% (más de un 90% es suficiente)

Ejercicio 6

PCA asume proyecciones LINEALES sobre hiperplanos. Si se aplica sobre estos datos 3D (rollo suizo) para hacerlos 2D, aplastará todo en una única espiral, y esa información de datos que están 'en el mismo punto de la espiral' pero lejos.

UMAP es el más adecuado porque lo que hace es aprender una función de 'relación'/mapeo y se puede usar con .transform de Scikit-Learn.

t-SNE no es paramétrico y optimiza las posiciones individualmente, pero NO PERMITE transformar datos nuevos que le pasemos, sin reentrenar todo otra vez.

Ejercicio 11

In [4]:
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier

# Generación de datos sintéticos: 500 muestras, 30 características (muchas redundantes)
X, y = make_classification(n_samples=500, n_features=30, n_informative=5, random_state=42)

# 1. Construye el Pipeline encadenando: Estandarización, PCA (retener 5 componentes) y RandomForest
pipeline = make_pipeline(
    StandardScaler(),
    PCA(n_components=5),
    RandomForestClassifier(n_estimators=50, random_state=42)
)

# 2. Evalúa el pipeline mediante validación cruzada de 5 pliegues midiendo la métrica 'accuracy'
scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')

print(f"Exactitud media tras hacer la validación cruzada: {scores.mean():.4f} (+/- {scores.std():.4f})")

Exactitud media tras hacer la validación cruzada: 0.7620 (+/- 0.0387)


Ejercicio 12

In [5]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Datos sintéticos no lineales (dos semicírculos)
X, y = make_moons(n_samples=400, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 1. Entrena K-Medias con K=5 clusters (supongamos que es lo ideal) sobre el conjunto de entrenamiento
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(X_train)

# 2. Genera las distancias de cada punto a los 5 centroides (transformación no supervisada)
X_train_dist = kmeans.transform(X_train)
X_test_dist = kmeans.transform(X_test)

# 3. Concatena horizontalmente (axis=1) las características originales con las distancias a los clusters
X_train_ext = np.hstack([X_train, X_train_dist])
X_test_ext = np.hstack([X_test, X_test_dist])

# 4. Entrena un clasificador supervisado lineal (Regresión Logística) sobre el espacio extendido
clf = LogisticRegression()
clf.fit(X_train_ext, y_train)
y_pred = clf.predict(X_test_ext)

print("Exactitud del modelo extendido gracias a ingeniería de características (hecha con agrupamiento K-Medias):", accuracy_score(y_test, y_pred))

Exactitud del modelo extendido gracias a ingeniería de características (hecha con agrupamiento K-Medias): 0.8583333333333333


Ejercicio 14

In [ ]:
import numpy as np
import sklearn.tree.DecisionTreeClassifier
from scipy.stats import mode
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.svm import SVC
from sklearn.metrics import classification_report

# Generamos 300 muestras, pero asumimos que solo conocemos las etiquetas del 10% (30 muestras)
X, y = make_classification(n_samples=300, n_classes=2, random_state=42)
X_labeled, y_labeled = X[:30], y[:30]
X_unlabeled = X[30:]

# 1. Agrupamos TODO el conjunto de datos X en K=10 clusters (habría que ver cuantos hacen faltan) usando K-Medias
kmeans = KMeans(n_clusters=10, random_state=42)
cluster_labels_all = kmeans.fit_predict(X)

# 2. Averiguamos la clase supervisada mayoritaria para cada uno de los 10 clusters
pseudo_labels_map = {}
cluster_labels_labeled = cluster_labels_all[:30]

for k in range(10):
    # Selecciona las etiquetas supervisadas reales de las muestras que cayeron en el cluster k
    y_in_cluster = y_labeled[cluster_labels_labeled == k]
    if len(y_in_cluster) > 0:
        # Asigna la moda de la clase al diccionario del cluster
        pseudo_labels_map[k] = mode(y_in_cluster, keepdims=False).mode
    else:
        pseudo_labels_map[k] = 0  # Valor por defecto (HABRÍA QUE VER CUAL ES EL MÁS RAZONABLE) si un cluster no tiene muestras etiquetadas

# 3. Propaga las pseudo-etiquetas a todas las muestras sin etiquetar
y_unlabeled_pseudo = np.array([pseudo_labels_map[c] for c in cluster_labels_all[30:]])

# 4. Une los datos etiquetados originales con los pseudo-etiquetados para entrenar un Árbol de Decisión
X_train_full = np.vstack([X_labeled, X_unlabeled])
y_train_full = np.concatenate([y_labeled, y_unlabeled_pseudo])

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train_full, y_train_full)
print("Árbol entrenado con dataset pseudo-etiquetado exitosamente.")